In [1]:
import pandas as pd
import numpy as np

In [2]:
import argparse
import os
import sys
import random
from loguru import logger
from tqdm import tqdm

# Add project root to path
try:
    from cpp.data_classes import Data
    from cpp.utils import SEP_TOKEN
    from cpp.data_loaders import load_job_skill_data_by_id
except ImportError as e:
    print(f"Error: Required modules not found. {e}")
    sys.exit(1)

In [9]:
# Set current directory to root of repo
import os
os.chdir('..')
print(f"Current working directory: {os.getcwd()}")


Current working directory: /dss/dsshome1/02/ra95kix2/thesis/skills4cpp


In [3]:
# --- Step 1: Load Data ---
data = Data(DATA_TYPE="decorte", ONLY_TITLES= False)
(train_pairs, train_job_ids), (val_pairs, val_job_ids), (test_pairs, test_job_ids) = data.get_data_with_job_ids(stage='transformation_finetuning')

# Collect all job IDs from data
all_job_ids_in_data = set()
for ids in train_job_ids + val_job_ids + test_job_ids:
    all_job_ids_in_data.update(str(jid) for jid in ids)
    
logger.info(f"  ✓ Found {len(all_job_ids_in_data)} unique job IDs across train/val/test")


Loading job_id lookup for decorte dataset...
  > Loaded job_id lookup with 9885 entries from /dss/dsshome1/02/ra95kix2/thesis/skills4cpp/data/title_pairs_desc/decorte_master_3.csv


FileNotFoundError: [Errno 2] No such file or directory: 'data/occupations_en.csv'

In [57]:
# Find missing values in all_job_ids_in_data (should be equal to range(7930))
expected_job_ids = set(str(i) for i in range(7930))
actual_job_ids = all_job_ids_in_data

missing_job_ids = expected_job_ids - actual_job_ids
extra_job_ids = actual_job_ids - expected_job_ids

print(f"Expected job IDs: {len(expected_job_ids)}")
print(f"Actual job IDs: {len(actual_job_ids)}")
print(f"Missing job IDs: {len(missing_job_ids)}")
print(f"Extra job IDs: {len(extra_job_ids)}")

if missing_job_ids:
    print(f"\nMissing job IDs (first 20): {sorted(missing_job_ids, key=int)[:20]}")
    
if extra_job_ids:
    print(f"\nExtra job IDs (first 20): {sorted(extra_job_ids, key=int)[:20]}")


Expected job IDs: 7930
Actual job IDs: 5802
Missing job IDs: 2128
Extra job IDs: 0

Missing job IDs (first 20): ['2', '7', '13', '20', '28', '31', '32', '38', '40', '43', '46', '47', '50', '52', '54', '58', '60', '62', '65', '71']


In [61]:
master_df.query('job_id == 46')

,Unnamed: 0,raw_title,raw_description,esco_title,esco_description,esco_id,job_id
46,46,Executive Chef/ Food Service Director,- Provide all phases of the hiring disciplinar...,head chef,Head chefs manage the kitchen to oversee the p...,http://data.europa.eu/esco/occupation/01484951...,46


In [51]:
tst = train_pairs[9054]
tst_ids = train_job_ids[9054]

In [52]:
tst_1 = tst[0]

In [53]:
tst_1.split('<SEP>')

['role: Adjunct Associate Professor/Academic Advisor \n description: Taught 6 classes on Organizational Behavior and Politics. Advised over 100 students.',
 'role: Graduation Auditor/Adjunct Assistant Professor \n description: Served as assistant registrar in addition to responsibilities of auditing every senior for suitability to graduate. Also taught courses as Visiting Assistant Professor.',
 'role: Director of Career Services \n description: Changed career services office to career development/service learning model based on Cognitive Information Processing model (one of only 16 such centers nationwide). Realized 40% increase in student usage of services and 67% increase in job placement. Developed a Leadership Fellows Program for national experiential learning opportunities. Changed curriculum to include course for rising sophomores/transitioning juniors: "Seminar on Career Development and Professionalism" combining both theory and extensive praxis. Received institutional recognit

In [54]:
tst_ids

['4195', '4196', '4197', '4198', '4199', '4200', '4201', '4202']

In [20]:
master_df = pd.read_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/data/title_pairs_desc/decorte_master.csv')

In [22]:
master_df.query('job_id in (7138, 7139, 7140)')

,Unnamed: 0,raw_title,raw_description,esco_title,esco_description,esco_id,job_id
7138,49,Internet Marketing Manager,Developed website content and directed PPC cam...,web content manager,Web content managers curate or create content ...,http://data.europa.eu/esco/occupation/a74f6c51...,7138
7139,50,Website Administrator,Updated and managed existing website propertie...,user interface designer,User interface designers are in charge of desi...,http://data.europa.eu/esco/occupation/96e20037...,7139
7140,51,Web Metrics Analyst,Developed metrics to identify inefficiencies a...,business analyst,Business analysts research and understand the ...,http://data.europa.eu/esco/occupation/60082a99...,7140


In [39]:
master_df.query('job_id == 11')

,Unnamed: 0,raw_title,raw_description,esco_title,esco_description,esco_id,job_id
11,11,Executive Chef / Event Consultant,Liaison to the Chicago Board of Realtors in co...,head chef,Head chefs manage the kitchen to oversee the p...,http://data.europa.eu/esco/occupation/01484951...,11


In [45]:
master_df.query('raw_title == "Consultant"')

,Unnamed: 0,raw_title,raw_description,esco_title,esco_description,esco_id,job_id
1534,1534,Consultant,Provided methodological training directly to t...,sociologist,Sociologists focus their research on explainin...,http://data.europa.eu/esco/occupation/11df8941...,1534
1669,1669,Consultant,Formulated strategic work plans and drafted fu...,public affairs consultant,Public affairs consultants function as represe...,http://data.europa.eu/esco/occupation/9e285322...,1669
1821,1821,Consultant,- Draft legal documentation for cross-border a...,legal consultant,Legal consultants advise a varied array of cli...,http://data.europa.eu/esco/occupation/31854d78...,1821
2639,2639,Consultant,Provided financial consulting services to Fort...,business consultant,"Business consultants analyse the position, str...",http://data.europa.eu/esco/occupation/6a29d804...,2639
2667,2667,Consultant,Handled the entire employment cycle from onboa...,human resources manager,"Human resources managers plan, design and impl...",http://data.europa.eu/esco/occupation/f605bcd2...,2667
2673,2673,Consultant,Helped customers select products that best fit...,customer service representative,Customer service representatives handle compla...,http://data.europa.eu/esco/occupation/13d1b2b4...,2673
2678,2678,Consultant,Provide consulting and technical training on p...,healthcare consultant,Healthcare consultants advise health care orga...,http://data.europa.eu/esco/occupation/61053604...,2678
2683,2683,Consultant,"- Track and maintain key dates, deadlines, and...",human resources officer,Human resources officers develop and implement...,http://data.europa.eu/esco/occupation/d3e32e5e...,2683
2693,2693,Consultant,Provide temporary medical social work services...,consultant social worker,Consultant social workers deliver high quality...,http://data.europa.eu/esco/occupation/186ef31e...,2693
2698,2698,Consultant,Translated observational data into user needs ...,web content manager,Web content managers curate or create content ...,http://data.europa.eu/esco/occupation/a74f6c51...,2698


In [50]:
# Find "consultant" role in train_pairs (first document)
consultant_indices = []
for i, pair in enumerate(train_pairs):
    first_doc = pair[0].split('<SEP>')  # Get first document from the pair
    for it in first_doc:
        if 'began consultancy as a professional and' in it.lower():
            consultant_indices.append(i)

print(f"Found {len(consultant_indices)} pairs with 'consultant' in first document")
if consultant_indices:
    print(f"First few indices: {consultant_indices[:5]}")
    # Show first example
    example_idx = consultant_indices[-1]
    print(f"\nExample at index {example_idx}:")
    print(train_pairs[example_idx][0].split('<SEP>'))


Found 14 pairs with 'consultant' in first document
First few indices: [9041, 9042, 9043, 9044, 9045]

Example at index 9054:
['role: Adjunct Associate Professor/Academic Advisor \n description: Taught 6 classes on Organizational Behavior and Politics. Advised over 100 students.', 'role: Graduation Auditor/Adjunct Assistant Professor \n description: Served as assistant registrar in addition to responsibilities of auditing every senior for suitability to graduate. Also taught courses as Visiting Assistant Professor.', 'role: Director of Career Services \n description: Changed career services office to career development/service learning model based on Cognitive Information Processing model (one of only 16 such centers nationwide). Realized 40% increase in student usage of services and 67% increase in job placement. Developed a Leadership Fellows Program for national experiential learning opportunities. Changed curriculum to include course for rising sophomores/transitioning juniors: "Sem

In [66]:
master_df['raw_desc_short'] = master_df.raw_description.apply(lambda x: x[:10] if pd.notna(x) else x)

In [65]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7930 entries, 0 to 7929
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        7930 non-null   int64 
 1   raw_title         7930 non-null   object
 2   raw_description   7922 non-null   object
 3   esco_title        7930 non-null   object
 4   esco_description  7928 non-null   object
 5   esco_id           7930 non-null   object
 6   job_id            7930 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 433.8+ KB


In [68]:
master_df.drop_duplicates(subset=['raw_title', 'raw_desc_short'])

,Unnamed: 0,raw_title,raw_description,esco_title,esco_description,esco_id,job_id,raw_desc_short
0,0,Line Cook,- Prepped food for the line and cooked appetiz...,cook,Cooks are culinary operatives who are able to ...,http://data.europa.eu/esco/occupation/90f75f67...,0,- Prepped
1,1,Line Chef,- Prepared food for the kitchen and created sp...,head chef,Head chefs manage the kitchen to oversee the p...,http://data.europa.eu/esco/occupation/01484951...,1,- Prepared
2,2,Executive Chef,- Promoted from rounds chef to sous chef in 20...,head chef,Head chefs manage the kitchen to oversee the p...,http://data.europa.eu/esco/occupation/01484951...,2,- Promoted
3,3,Banquet Cook 2( Lead Cook),- Attended daily BEO meetings\n- Prepped and f...,cook,Cooks are culinary operatives who are able to ...,http://data.europa.eu/esco/occupation/90f75f67...,3,- Attended
4,4,Kitchen Supervisor,- Oversaw the am shift\n- Part of a renovation...,restaurant manager,Restaurant managers are in charge of managing ...,http://data.europa.eu/esco/occupation/d5eb6150...,4,- Oversaw
...,...,...,...,...,...,...,...,...
7925,836,Intraday Operations Analyst,Supervised a team responsible for intraday wor...,call centre supervisor,Call centre supervisors oversee call centre em...,http://data.europa.eu/esco/occupation/6aa78e46...,7925,Supervised
7926,837,Consultant,Supervised operations and project implementati...,business service manager,Business service managers are reponsible for t...,http://data.europa.eu/esco/occupation/3c136897...,7926,Supervised
7927,838,Workforce Specialist (WFM),Ensured optimum intraday staffing and performa...,contact centre supervisor,Contact centre supervisors oversee and coordin...,http://data.europa.eu/esco/occupation/9d2aae3e...,7927,Ensured op
7928,839,Sr. Workforce Manager,Reduced non-productive agent time by 33% for a...,contact centre manager,Contact centre managers coordinate and plan th...,http://data.europa.eu/esco/occupation/f5166c6b...,7928,Reduced no


In [70]:
tst

('role: Adjunct Associate Professor/Academic Advisor \n description: Taught 6 classes on Organizational Behavior and Politics. Advised over 100 students.<SEP>role: Graduation Auditor/Adjunct Assistant Professor \n description: Served as assistant registrar in addition to responsibilities of auditing every senior for suitability to graduate. Also taught courses as Visiting Assistant Professor.<SEP>role: Director of Career Services \n description: Changed career services office to career development/service learning model based on Cognitive Information Processing model (one of only 16 such centers nationwide). Realized 40% increase in student usage of services and 67% increase in job placement. Developed a Leadership Fellows Program for national experiential learning opportunities. Changed curriculum to include course for rising sophomores/transitioning juniors: "Seminar on Career Development and Professionalism" combining both theory and extensive praxis. Received institutional recognit

In [71]:
import json

# Load the fused scores
with open('/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/outputs/linear_fusion_sum/best_fused_scores.json', 'r') as f:
    fused_scores = json.load(f)


In [72]:
all_keys = fused_scores['scores'].keys()

In [73]:
# Find missing values in all_job_ids_in_data (should be equal to range(7930))
expected_job_ids = set(str(i) for i in range(7930))
actual_job_ids = all_keys

missing_job_ids = expected_job_ids - actual_job_ids
extra_job_ids = actual_job_ids - expected_job_ids

print(f"Expected job IDs: {len(expected_job_ids)}")
print(f"Actual job IDs: {len(actual_job_ids)}")
print(f"Missing job IDs: {len(missing_job_ids)}")
print(f"Extra job IDs: {len(extra_job_ids)}")

if missing_job_ids:
    print(f"\nMissing job IDs (first 20): {sorted(missing_job_ids, key=int)[:20]}")
    
if extra_job_ids:
    print(f"\nExtra job IDs (first 20): {sorted(extra_job_ids, key=int)[:20]}")


Expected job IDs: 7930
Actual job IDs: 7930
Missing job IDs: 0
Extra job IDs: 0


In [1]:
import pandas as pd
df = pd.read_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/experiments/consolidated_results.csv')

In [2]:
df.columns

Index(['experiment_name', 'model_id', 'dataset', 'recall@1', 'recall@5',
       'recall@10', 'map@10', 'mrr@10', 'map_full', 'mrr_full', 'coverage',
       'N_eval', 'encode_ms_per_query', 'skill_coverage@1', 'skill_coverage@3',
       'skill_coverage@5', 'skill_coverage@10', 'topk', 'use_faiss',
       'normalize_embeddings'],
      dtype='object')

In [3]:
df.sort_values('recall@5')

,experiment_name,model_id,dataset,recall@1,recall@5,recall@10,map@10,mrr@10,map_full,mrr_full,coverage,N_eval,encode_ms_per_query,skill_coverage@1,skill_coverage@3,skill_coverage@5,skill_coverage@10,topk,use_faiss,normalize_embeddings
13,local_test,all-MiniLM-L6-v2,decorte,0.165132,0.355774,0.456322,0.247110,0.247110,NaN,NaN,0.999502,16080,0.349486,NaN,NaN,NaN,NaN,10,False,True
7,infer_kw_cp_test_pjmath_1,JobGTE-7b-Lora,karrierewege_plus_cp_test_pairs,0.183342,0.422254,0.531481,0.285834,0.285834,0.298994,0.298994,0.999868,15154,5.848594,NaN,NaN,NaN,NaN,10,True,True
9,infer_kw_occ_test_pjmath_1,JobGTE-7b-Lora,karrierewege_plus_occ_test_pairs,0.196145,0.469104,0.593537,0.310996,0.310996,0.324787,0.324787,0.999150,3531,5.979895,NaN,NaN,NaN,NaN,10,True,True
0,infer_decorte_all_pjmath_1,JobGTE-7b-Lora,decorte,0.263749,0.501009,0.591196,0.364779,0.364779,0.375566,0.375566,0.999748,7930,6.833624,0.381793,0.574913,0.653667,0.747272,10,True,True
1,infer_decorte_all_pjmath_1_v2,JobGTE-7b-Lora,decorte,0.263749,0.501009,0.591196,0.364779,0.364779,0.375566,0.375566,0.999748,7930,6.862265,0.381793,0.574913,0.653667,0.747272,10,True,True
6,infer_kw_cp_all_techwolf,JobBERT-v2,karrierewege_plus_cp_test_pairs,0.243662,0.507974,0.616404,0.355694,0.355694,0.367581,0.367581,1.000000,97196,0.261949,0.395431,0.601366,0.684275,0.778349,10,True,True
3,infer_decorte_test_pjmath_1,JobGTE-7b-Lora,decorte,0.280435,0.532609,0.623913,0.385901,0.385901,0.395377,0.395377,0.993521,926,7.095738,0.393001,0.596707,0.677876,0.768098,10,True,True
8,infer_kw_occ_all_techwolf,JobBERT-v2,karrierewege_plus_occ_test_pairs,0.237254,0.541769,0.668612,0.367396,0.367396,0.379531,0.379531,1.000000,13024,0.332924,0.401048,0.631395,0.719132,0.823667,10,True,True
2,infer_decorte_all_techwolf,JobBERT-v2,decorte,0.327195,0.596998,0.687815,0.443160,0.443160,0.452277,0.452277,0.999748,7930,0.282690,0.459434,0.661742,0.734047,0.816467,10,True,True
5,infer_decorte_test_techwolf_expanded,JobBERT-v2,decorte,0.317838,0.616216,0.711351,0.442363,0.442363,0.451138,0.451138,0.998920,926,0.247481,0.450077,0.668958,0.747135,0.829699,10,True,True


In [1]:
import pandas as pd

In [2]:
df_vanilla = pd.read_csv(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/vanilla_ir_eval_results.csv')
df_ir = pd.read_csv(r'/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/results/ir_eval_results.csv')

ParserError: Error tokenizing data. C error: Expected 13 fields in line 5, saw 22


In [11]:
df_vanilla.query('metric == "mrr@full" and dataset == "decorte"')

,model_id,task,dataset,use_alias_expansion,full_metric,metric,value
15,TechWolf/JobBERT-v2,B,decorte,True,_TaskB_cosine_mrr@149695,mrr@full,0.460059
66,pj-mathematician/JobSkillGTE-7b-lora,B,decorte,True,_TaskB_cosine_mrr@149695,mrr@full,0.379490
100,pj-mathematician/JobSkillBGE-large-en-v1.5,B,decorte,True,_TaskB_cosine_mrr@149695,mrr@full,0.420619


In [ ]:
 # "recall@10": 0.06287985381579954,


In [1]:
import pandas as pd
from pathlib import Path

# Load the data files
data_dir = Path('../data/talent_clef/TaskA/validation/english')

# Load queries (TSV file with header)
queries_df = pd.read_csv(data_dir / 'queries', sep='\t')
queries_df.columns = ['query_id', 'query_text']

# Load corpus (TSV file with header)
corpus_df = pd.read_csv(data_dir / 'corpus_elements', sep='\t')
corpus_df.columns = ['corpus_id', 'corpus_text']

# Load qrels (relevance judgments) - TSV file
qrels_df = pd.read_csv(data_dir / 'qrels.tsv', sep='\t', header=None, names=['query_id', 'iteration', 'corpus_id', 'relevance'])

# Merge qrels with queries
merged_df = qrels_df.merge(queries_df, on='query_id', how='left')

# Merge with corpus
result_df = merged_df.merge(corpus_df, on='corpus_id', how='left')

# Select and reorder columns
result_df = result_df[['query_id', 'query_text', 'corpus_id', 'corpus_text', 'relevance']]

print(f"Total rows: {len(result_df)}")
print(f"Unique queries: {result_df['query_id'].nunique()}")
print(f"\nFirst few rows:")
result_df.head(10)


Total rows: 2420
Unique queries: 105

First few rows:


,query_id,query_text,corpus_id,corpus_text,relevance
0,1,nanny,143,counselor,1
1,1,nanny,150,daycare teacher,1
2,1,nanny,764,assistant preschool teacher,1
3,1,nanny,870,babysitter,1
4,1,nanny,1464,classroom assistant,1
5,1,nanny,1488,childcare assistant,1
6,1,nanny,1580,corps member,1
7,1,nanny,1676,childcare provider,1
8,1,nanny,1961,student placement coordinator,1
9,1,nanny,1963,childcare worker,1


In [2]:
result_df.tail(10)

,query_id,query_text,corpus_id,corpus_text,relevance
2410,105,publisher,1867,travel editor,1
2411,105,publisher,1996,editor in chief,1
2412,105,publisher,2010,group publisher,1
2413,105,publisher,2012,editorial director,1
2414,105,publisher,2092,commissioning editor,1
2415,105,publisher,2123,copywriter,1
2416,105,publisher,2144,head of publishing,1
2417,105,publisher,2356,editorial manager,1
2418,105,publisher,2400,editorial supervisor,1
2419,105,publisher,2455,chief reporter,1


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("TechWolf/JobBERT-v2")

In [4]:
print(model)

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Router(
    (sub_modules): ModuleDict(
      (anchor): Sequential(
        (0): Dense({'in_features': 768, 'out_features': 1024, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
      )
      (positive): Sequential(
        (0): Dense({'in_features': 768, 'out_features': 1024, 'bias': True, 'activation_function': 'torch.nn.modules.activation.Tanh'})
      )
    )
  )
)


In [33]:
v1 = model.encode('publisher', prompt_name='query')

In [34]:
v2 = model.encode('travel editor', prompt_name='document')

In [36]:
v_esco = model.encode('publisher', prompt_name="esco")

ValueError: Prompt name 'esco' not found in the configured prompts dictionary with keys ['query', 'document'].

In [35]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Reshape vectors for cosine_similarity function
v1_reshaped = v1.reshape(1, -1)
v2_reshaped = v2.reshape(1, -1)

similarity = cosine_similarity(v1_reshaped, v2_reshaped)[0][0]
print(f"Cosine similarity between 'publisher' and 'travel editor': {similarity:.4f}")

Cosine similarity between 'publisher' and 'travel editor': 0.5722


In [ ]:
import torch

sentences = ["This is an example sentence."]

# 1. Get the standard embeddings (Base + Pooling)
# We use output_value="sentence_embedding" to get the 768-dim vector
embeddings = model.encode(sentences, convert_to_tensor=True)

# 2. Pass through the 'anchor' head (result: 1024-dim)
anchor_embeddings = model[2].sub_modules['anchor'](embeddings)

# 3. Pass through the 'positive' head (result: 1024-dim)
positive_embeddings = model[2].sub_modules['positive'](embeddings)

In [39]:
import torch
import numpy as np
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device, cos_sim

# Load the model
model = SentenceTransformer("TechWolf/JobBERT-v2")

def encode_batch(jobbert_model, texts, key="anchor"):
    features = jobbert_model.tokenize(texts)
    features = batch_to_device(features, jobbert_model.device)
    features["text_keys"] = [key]
    with torch.no_grad():
        out_features = jobbert_model.forward(features)
    return out_features["sentence_embedding"].cpu().numpy()

def encode(jobbert_model, texts, key="anchor", batch_size: int = 8):
    # Sort texts by length and keep track of original indices
    sorted_indices = np.argsort([len(text) for text in texts])
    sorted_texts = [texts[i] for i in sorted_indices]
    
    embeddings = []
    
    # Encode in batches
    for i in tqdm(range(0, len(sorted_texts), batch_size)):
        batch = sorted_texts[i:i+batch_size]
        embeddings.append(encode_batch(jobbert_model, batch, key))
    
    # Concatenate embeddings and reorder to original indices
    sorted_embeddings = np.concatenate(embeddings)
    original_order = np.argsort(sorted_indices)
    return sorted_embeddings[original_order]

# Example usage
job_titles = [
    'Software Engineer',
    'Senior Software Developer',
    'Product Manager',
    'Data Scientist'
]

# Get embeddings
embeddings = encode(model, job_titles)

# Calculate cosine similarity matrix
similarities = cos_sim(embeddings, embeddings)
print(similarities)


  0%|          | 0/1 [00:00<?, ?it/s]

tensor([[1.0000, 0.8723, 0.4821, 0.5447],
        [0.8723, 1.0000, 0.4822, 0.5019],
        [0.4821, 0.4822, 1.0000, 0.4328],
        [0.5447, 0.5019, 0.4328, 1.0000]])


In [40]:
result_df.shape

(2420, 5)

In [43]:
result_df.query("relevance == 1").groupby('corpus_text').query_text.nunique().sort_values()

corpus_text
network control technician        1
network security administrator    1
network provisioner               1
network planning engineer         1
network performance engineer      1
                                 ..
filmmaker                         3
film editor                       3
cinematographer                   3
configuration technician          3
film and video editor             3
Name: query_text, Length: 2107, dtype: int64

In [44]:
result_df.query("relevance == 1").groupby('query_text').corpus_text.nunique().sort_values()

query_text
biomedical engineer            1
acupuncturist                  2
agronomist                     2
metallurgist                   2
interior decorator             3
                              ..
public relations executive    47
technical recruiter           48
lawyer                        49
banker                        51
talent acquisition            53
Name: corpus_text, Length: 105, dtype: int64

In [2]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import batch_to_device, cos_sim

model = SentenceTransformer("TechWolf/JobBERT-v2")

def encode_with_head(jobbert_model, texts, text_key="anchor", batch_size=8):
    """
    Encode texts using specified projection head.
    
    Args:
        text_key: "anchor" or "positive" to select projection head
    """
    sorted_indices = np.argsort([len(text) for text in texts])
    sorted_texts = [texts[i] for i in sorted_indices]
    
    embeddings = []
    
    for i in range(0, len(sorted_texts), batch_size):
        batch = sorted_texts[i:i+batch_size]
        
        features = jobbert_model.tokenize(batch)
        features = batch_to_device(features, jobbert_model.device)
        features["text_keys"] = [text_key]  # Specify projection head here!
        
        with torch.no_grad():
            out_features = jobbert_model.forward(features)
        
        embeddings.append(out_features["sentence_embedding"].cpu().numpy())
    
    sorted_embeddings = np.concatenate(embeddings)
    original_order = np.argsort(sorted_indices)
    return sorted_embeddings[original_order]

# For free-text job titles (queries)
job_titles = ['travel editor', 'software engineer', 'data scientist']
job_embeddings = encode_with_head(model, job_titles, text_key="anchor")

# For ESCO occupation titles (documents to match against)
esco_titles = ['publisher', 'software developer', 'statistician']
esco_embeddings = encode_with_head(model, esco_titles, text_key="positive")

# Now compute cross-similarity (job titles × ESCO titles)
similarities = cos_sim(job_embeddings, esco_embeddings)
print(similarities)

tensor([[0.5722, 0.1269, 0.1956],
        [0.2914, 0.9141, 0.3620],
        [0.2671, 0.5249, 0.6417]])


In [3]:
# Check max context length of JobBERT
print(f"Max sequence length: {model.max_seq_length}")
print(f"Tokenizer max length: {model.tokenizer.model_max_length}")

# Check the actual model config
if hasattr(model[0], 'auto_model'):
    print(f"Model config max position embeddings: {model[0].auto_model.config.max_position_embeddings}")


Max sequence length: 512
Tokenizer max length: 512
Model config max position embeddings: 514


In [4]:
# Check special tokens
print(f"Special tokens: {model.tokenizer.special_tokens_map}")
print(f"All special tokens: {model.tokenizer.all_special_tokens}")
print(f"All special IDs: {model.tokenizer.all_special_ids}")


Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '[UNK]', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
All special tokens: ['<s>', '</s>', '[UNK]', '<pad>', '<mask>']
All special IDs: [0, 2, 104, 1, 30526]


In [2]:
from sentence_transformers import SentenceTransformer

# Load the JobGTE-7b-Lora model

import os
os.environ['HF_HOME'] = "/dss/dssmcmlfs01/pr74ze/pr74ze-dss-0001/ra95kix2/.cache/huggingface"

# Load the JobGTE-7b-Lora model using its HuggingFace name
jobgte_model = SentenceTransformer("pj-mathematician/JobGTE-7b-Lora")

print(f"Max sequence length: {jobgte_model.max_seq_length}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/317 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/767 [00:00<?, ?B/s]

...

model-00004-of-00007.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

model-00006-of-00007.safetensors:   0%|          | 0.00/3.66G [00:00<?, ?B/s]

model-00005-of-00007.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00007.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

: 

In [1]:
import pandas as pd

df = pd.read_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/data/title_pairs_desc/category_test_split_isco.csv')

In [2]:
df.shape

(1046, 12)

In [10]:
import pandas as pd

df = pd.read_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/reports/aggregated_skill_metrics.csv')

In [11]:
df.query("folder.str.contains('isco')").dropna(axis=1).T

,0,1,4,9,10,14,18,19,21,22,25,28,34,35,37,38
folder,isco_model_h2_soft_deep_larger_val_mpnet_kw_cp,isco_model_h2_soft_deep_larger_val_mpnet_expan...,isco_model_h2_soft_deep_larger_val_mpnet_2,isco_model_h2_soft_deep_val_desc_jobbert,isco_model_h2_soft_deep_larger_val_desc_jobbert,isco_model_h3_soft_deep_larger_val_mpnet,isco_model_h4_soft_deep_larger_val_mpnet,isco_model_h2_soft_deep_larger_val_desc_jobber...,isco_model_h1_soft_deep_larger_val_mpnet,isco_model_h2_soft_deep_larger_val,isco_model_h2_soft_deep_larger_val_mpnet_kw_cp...,isco_model_h2_soft_deep_val_jobbert,isco_model_h2_soft_deep_larger_val_mpnet_expanded,isco_model_h2_soft_deep_larger_val_mpnet_wo_desc,isco_model_h2_soft_deep_larger_val_mpnet,isco_model_h2_soft_deep_larger_val_jobbert
file_name,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json,results.json
best_params.batch_size,256.0,256.0,64.0,32.0,64.0,32.0,64.0,128.0,64.0,256.0,128.0,64.0,256.0,64.0,32.0,64.0
best_params.dropout,0.15,0.5,0.2,0.3,0.5,0.35,0.3,0.2,0.25,0.45,0.25,0.3,0.15,0.15,0.15,0.0
best_params.hidden_dim_layer_0,768.0,1536.0,1024.0,768.0,2048.0,1536.0,2048.0,1024.0,2048.0,2048.0,512.0,1024.0,1536.0,768.0,1024.0,768.0
best_params.lr,0.000091,0.000033,0.000278,0.00018,0.000114,0.000163,0.000722,0.000012,0.000285,0.000056,0.003466,0.000127,0.000335,0.000443,0.000077,0.000029
best_params.n_layers,1.0,1.0,1.0,2.0,2.0,3.0,1.0,1.0,3.0,3.0,3.0,1.0,1.0,1.0,2.0,1.0
best_params.scheduler_factor,0.3,0.5,0.5,0.5,0.1,0.4,0.5,0.5,0.2,0.4,0.1,0.5,0.3,0.4,0.5,0.4
best_params.scheduler_patience,3.0,4.0,4.0,4.0,2.0,5.0,3.0,4.0,3.0,5.0,5.0,5.0,4.0,4.0,5.0,2.0
best_params.use_batchnorm,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


In [15]:
cols = [col for col in df.columns if 'accuracy' in col and 'val' not in col]

df.query("folder.str.contains('isco')").dropna(axis=1).set_index('folder').loc[:,cols].sort_values('test_metrics.accuracy').to_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/reports/aggregated_isco_model_test.csv')

In [13]:
df.query("folder.str.contains('isco') and folder.str.contains('desc')").dropna(axis=1).set_index('folder').loc[:,cols].sort_values('test_metrics.accuracy')

,test_metrics.accuracy,test_metrics.accuracy_top1,test_metrics.accuracy_top10,test_metrics.accuracy_top3,test_metrics.accuracy_top5
folder,,,,,
isco_model_h2_soft_deep_larger_val_mpnet_expanded_wo_desc,0.450402,0.450402,0.911528,0.710456,0.820375
isco_model_h2_soft_deep_larger_val_desc_jobbert_kw_cp,0.469169,0.469169,0.943700,0.761394,0.852547
isco_model_h2_soft_deep_larger_val_mpnet_wo_desc,0.490617,0.490617,0.898123,0.723861,0.809651
isco_model_h2_soft_deep_val_desc_jobbert,0.506702,0.506702,0.959786,0.793566,0.900804
isco_model_h2_soft_deep_larger_val_desc_jobbert,0.517426,0.517426,0.973190,0.820375,0.903485


In [16]:
df.query("folder.str.contains('isco')").dropna(axis=1).set_index('folder').loc[:,cols].sort_values('test_metrics.accuracy')

,test_metrics.accuracy,test_metrics.accuracy_top1,test_metrics.accuracy_top10,test_metrics.accuracy_top3,test_metrics.accuracy_top5
folder,,,,,
isco_model_h2_soft_deep_larger_val_mpnet_kw_cp,0.404826,0.404826,0.951743,0.734584,0.860590
isco_model_h2_soft_deep_larger_val_mpnet_kw_cp_decorte,0.404826,0.404826,0.946381,0.742627,0.833780
isco_model_h2_soft_deep_larger_val_mpnet_expanded_wo_desc,0.450402,0.450402,0.911528,0.710456,0.820375
isco_model_h2_soft_deep_larger_val_desc_jobbert_kw_cp,0.469169,0.469169,0.943700,0.761394,0.852547
isco_model_h2_soft_deep_larger_val,0.479893,0.479893,0.973190,0.774799,0.906166
isco_model_h2_soft_deep_larger_val_jobbert,0.482574,0.482574,0.927614,0.729223,0.844504
isco_model_h2_soft_deep_larger_val_mpnet_wo_desc,0.490617,0.490617,0.898123,0.723861,0.809651
isco_model_h4_soft_deep_larger_val_mpnet,0.498660,0.498660,0.914209,0.734584,0.847185
isco_model_h2_soft_deep_val_jobbert,0.498660,0.498660,0.943700,0.750670,0.847185


In [6]:
import pandas as pd

df_occ = pd.read_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/data/esco_datasets/occupations_en.csv')
df_rel = pd.read_csv(r'/dss/dsshome1/02/ra95kix2/thesis/skills4cpp/data/esco_datasets/occupationSkillRelations_en.csv')

In [8]:
df_occ.merge(df_rel, how='right', left_on='conceptUri', right_on='occupationUri').groupby('iscoGroup').skillUri.nunique()

iscoGroup
110      98
210      56
310     196
1111     62
1112    177
       ... 
9613     26
9621     21
9622     49
9623     41
9629    125
Name: skillUri, Length: 426, dtype: int64

In [9]:
df_occ.merge(df_rel, how='right', left_on='conceptUri', right_on='occupationUri').iscoGroup.nunique()

426